[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_04_data_science_math/ch20_biostatistics_inference/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 20 Walkthrough: Probability, Statistical Inference, Hypothesis Testing & Epidemiological Metrics

**Book Reference:** Chapter 20 (Part IV Capstone), Sections 20.1 – 20.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every analysis uses the bundled `mch_survey_sample` maternal and child health dataset.

In [ ]:
# @title Step 0: Click Play (▶) to Load Packages & MCH Survey Data
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import math
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import sharukh_tutor as tutor

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
mch = mch_raw.copy()
mch["residence"] = mch["residence"].astype(str).str.strip().str.title()
mch["anc_visits"] = mch["anc_visits"].replace(999, np.nan)
print("Loaded mch_survey_sample:", mch.shape)
print(mch["residence"].value_counts().to_dict())


## Section 20.3.1 — Probability Distributions & CLT Monte Carlo Simulation

In [ ]:
# 1. Theoretical Normal & Binomial probabilities
p_anemic_hb = stats.norm.cdf(11.0, loc=11.3, scale=1.4)
p_20_or_more = stats.binom.sf(19, n=50, p=0.35)
print(f"P(Hb < 11.0 g/dL | N(11.3, 1.4)): {p_anemic_hb:.4f}")
print(f"P(X >= 20 cases | Binom(50, 0.35)): {p_20_or_more:.4f}")

# 2. Central Limit Theorem (CLT) from skewed Exponential(scale=2.0)
rng = np.random.default_rng(42)
draws = rng.exponential(scale=2.0, size=(2000, 30))
sample_means = draws.mean(axis=1)
theo_se = 2.0 / math.sqrt(30)
print(f"CLT mean of 2,000 means: {sample_means.mean():.4f} (true=2.0)")
print(f"Empirical SE: {sample_means.std(ddof=1):.4f} "
      f"| Theoretical SE: {theo_se:.4f}")


## Section 20.3.2 — Proportion Estimation & 95% Wilson Score CI

In [ ]:
def wilson_ci_demo(successes, n, z=1.96):
    p = successes / n
    denom = 1 + (z ** 2) / n
    center = (p + (z ** 2) / (2 * n)) / denom
    margin = (
        z * math.sqrt((p * (1 - p) + (z ** 2) / (4 * n)) / n)
    ) / denom
    return (round(p, 4), round(max(0.0, center - margin), 4),
            round(min(1.0, center + margin), 4))

anemic_n = int(mch["maternal_anemia"].sum())
total_n = len(mch)
prop, lo, hi = wilson_ci_demo(anemic_n, total_n)
print(f"Maternal anemia: {anemic_n}/{total_n} = {prop:.4f} "
      f"(95% Wilson CI: [{lo:.4f}, {hi:.4f}])")
print("Zero-event sample (0/20) Wilson CI:", wilson_ci_demo(0, 20))


## Section 20.3.3 — Parametric Comparison: Welch's t-Test & Pooled Cohen's d

In [ ]:
hb_urban = mch.loc[mch["residence"] == "Urban", "maternal_hb_g_dl"]
hb_rural = mch.loc[mch["residence"] == "Rural", "maternal_hb_g_dl"]
a = hb_urban.dropna().to_numpy()
b = hb_rural.dropna().to_numpy()

t_res = stats.ttest_ind(a, b, equal_var=False)
var_a, var_b = np.var(a, ddof=1), np.var(b, ddof=1)
pooled_sd = math.sqrt(
    ((len(a) - 1) * var_a + (len(b) - 1) * var_b) / (len(a) + len(b) - 2)
)
cohens_d = (a.mean() - b.mean()) / pooled_sd
print(f"Urban Hb mean: {a.mean():.3f} (n={len(a)}) | "
      f"Rural Hb mean: {b.mean():.3f} (n={len(b)})")
print(f"Welch t={t_res.statistic:.4f}, p={t_res.pvalue:.4f}, "
      f"Cohen's d={cohens_d:.4f}")


## Section 20.3.4 — Nonparametric Rank Test: Mann-Whitney U on Skewed ANC Visits

In [ ]:
anc_urban = mch.loc[mch["residence"] == "Urban", "anc_visits"].dropna()
anc_rural = mch.loc[mch["residence"] == "Rural", "anc_visits"].dropna()

u_res = stats.mannwhitneyu(anc_urban, anc_rural, alternative="two-sided")
print(f"Median ANC visits — Urban: {anc_urban.median():.1f}, "
      f"Rural: {anc_rural.median():.1f}")
print(f"Mann-Whitney U={u_res.statistic:.2f}, "
      f"p={u_res.pvalue:.6f}, significant={bool(u_res.pvalue < 0.05)}")


## Section 20.3.5 — Categorical Association: Pearson Chi-Square & Fisher's Exact Test

In [ ]:
ct = pd.crosstab(mch["residence"], mch["maternal_anemia"])
a_cell = int(ct.loc["Rural", 1])
b_cell = int(ct.loc["Rural", 0])
c_cell = int(ct.loc["Urban", 1])
d_cell = int(ct.loc["Urban", 0])
table_2x2 = [[a_cell, b_cell], [c_cell, d_cell]]

chi2, p_chi2, dof, expected = stats.chi2_contingency(
    table_2x2, correction=False
)
print("2x2 Table (Rural vs Urban x Anemia):", table_2x2)
print(f"Pearson chi2={chi2:.4f}, p={p_chi2:.4f}, "
      f"dof={dof}, expected_min={expected.min():.2f}")

# Small-cell subgroup where expected_min < 5 -> use Fisher's Exact
sparse_tbl = [[1, 14], [7, 8]]
_, _, _, exp_sparse = stats.chi2_contingency(sparse_tbl, correction=False)
fish_res = stats.fisher_exact(sparse_tbl)
print(f"Sparse expected_min={exp_sparse.min():.2f} -> "
      f"Fisher OR={fish_res.statistic:.3f}, p={fish_res.pvalue:.4f}")


## Section 20.3.6 — Epidemiological Metrics: Risk Ratio & Woolf Odds Ratio 95% CI

In [ ]:
risk_rural = a_cell / (a_cell + b_cell)
risk_urban = c_cell / (c_cell + d_cell)
risk_ratio = risk_rural / risk_urban

odds_ratio = (a_cell * d_cell) / (b_cell * c_cell)
se_log_or = math.sqrt(1/a_cell + 1/b_cell + 1/c_cell + 1/d_cell)
ci_low = math.exp(math.log(odds_ratio) - 1.96 * se_log_or)
ci_high = math.exp(math.log(odds_ratio) + 1.96 * se_log_or)

print(f"Risk (Rural): {risk_rural:.4f} | Risk (Urban): {risk_urban:.4f}")
print(f"Risk Ratio (RR): {risk_ratio:.3f}")
print(f"Crude OR: {odds_ratio:.3f} "
      f"(95% Woolf CI: [{ci_low:.3f}, {ci_high:.3f}])")


## Section 20.3.7 — Multiple Hypothesis Testing & Benjamini-Hochberg FDR

In [ ]:
raw_p = [0.004, 0.032, 0.048, 0.210, 0.650]
bh_q = stats.false_discovery_control(raw_p, method="bh")
bonf_p = [min(1.0, p * len(raw_p)) for p in raw_p]

audit = pd.DataFrame({
    "raw_p": raw_p,
    "bh_q": np.round(bh_q, 4),
    "bonferroni_p": np.round(bonf_p, 4),
    "survives_fdr_05": bh_q < 0.05,
})
print(audit.to_string(index=False))


## Section 20.3.8 — Multivariable Logistic Regression (`statsmodels.api.Logit`)

In [ ]:
mch["is_urban"] = (mch["residence"] == "Urban").astype(int)
model_df = mch[["maternal_anemia", "anc_visits", "is_urban"]].dropna()
X = sm.add_constant(model_df[["anc_visits", "is_urban"]])
y = model_df["maternal_anemia"]

logit_fit = sm.Logit(y, X).fit(disp=False)
conf = logit_fit.conf_int()
aor_table = pd.DataFrame({
    "aor": np.exp(logit_fit.params),
    "ci_low": np.exp(conf[0]),
    "ci_high": np.exp(conf[1]),
    "p_value": logit_fit.pvalues,
}).round(4)
print(f"Complete cases fitted: n={len(model_df)}")
print(aor_table)


## Section 20.3.9 — Break It On Purpose! (Biostatistics Error Anatomy)

In [ ]:
# Intentional Bug #1: ValueError — negative count in contingency table
bad_table = [[-5, 20], [15, 30]]
stats.chi2_contingency(bad_table, correction=False)


In [ ]:
# Intentional Bug #2: ZeroDivisionError — Woolf SE with a zero cell
a0, b0, c0, d0 = 12, 38, 0, 50
se_broken = math.sqrt(1 / a0 + 1 / b0 + 1 / c0 + 1 / d0)


In [ ]:
# Intentional Bug #3: MissingDataError — passing NaN into sm.Logit
X_with_nan = sm.add_constant(mch[["anc_visits", "is_urban"]])
sm.Logit(mch["maternal_anemia"], X_with_nan).fit(disp=False)


## Section 20.4 — Stage 4 Blueprint: Publication-Ready Epidemiological Evidence Pipeline

In [ ]:
def build_mch_evidence_summary(raw_df):
    """Run end-to-end prevalence, wealth-gradient & adjusted logit."""
    df = raw_df.copy()
    df["residence"] = df["residence"].astype(str).str.strip().str.title()
    df["anc_visits"] = df["anc_visits"].replace(999, np.nan)
    df["is_urban"] = (df["residence"] == "Urban").astype(int)

    # 1. Overall anemia prevalence + 95% Wilson CI
    prev = wilson_ci_demo(int(df["maternal_anemia"].sum()), len(df))

    # 2. Hemoglobin across 5 wealth quintiles (ANOVA + Kruskal-Wallis)
    groups = [
        grp["maternal_hb_g_dl"].dropna().to_numpy()
        for _, grp in df.groupby("wealth_quintile")
    ]
    f_stat, anova_p = stats.f_oneway(*groups)
    h_stat, kruskal_p = stats.kruskal(*groups)

    # 3. Multivariable logistic regression (aOR table)
    m_df = df[["maternal_anemia", "anc_visits", "is_urban"]].dropna()
    X_m = sm.add_constant(m_df[["anc_visits", "is_urban"]])
    fit = sm.Logit(m_df["maternal_anemia"], X_m).fit(disp=False)
    ci = fit.conf_int()
    aor_df = pd.DataFrame({
        "aor": np.exp(fit.params),
        "ci_low": np.exp(ci[0]),
        "ci_high": np.exp(ci[1]),
        "p_value": fit.pvalues,
    }).round(4)
    wealth_tuple = (
        round(float(f_stat), 4), round(float(anova_p), 4),
        round(float(h_stat), 4), round(float(kruskal_p), 4),
    )
    return prev, wealth_tuple, aor_df

prev_ci, wealth_stats, aor_res = build_mch_evidence_summary(mch_raw)
print("Anemia Wilson CI:", prev_ci)
print("Wealth Hb (F, p, H, p):", wealth_stats)
print(aor_res)
